# 10x human–mouse mixture reproduction

This notebook downloads the native sample-filtered feature-barcode H5 from the [10x Genomics 10k HGMM dataset](https://www.10xgenomics.com/datasets/10k-hgmm-3p-gemx), separates the human and mouse cells, trains Xenocomm, and recreates the core analyses.


In [ ]:
from pathlib import Path
from shutil import copyfileobj
from urllib.request import Request, urlopen

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse
import xenocomm as xc

sns.set_theme(context="notebook", style="whitegrid")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
DATA_DIR = NOTEBOOK_DIR / "data/10x_hgmm"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs/10x_hgmm"
INPUT_H5 = DATA_DIR / "10k_hgmm_3p_gemx_count_sample_filtered_feature_bc_matrix.h5"
DATA_URL = (
    "https://cf.10xgenomics.com/samples/cell-exp/8.0.0/"
    "10k_hgmm_3p_gemx_10k_hgmm_3p_gemx/"
    "10k_hgmm_3p_gemx_10k_hgmm_3p_gemx_count_sample_filtered_feature_bc_matrix.h5"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not INPUT_H5.exists():
    print("Downloading the 10x sample-filtered feature-barcode matrix...")
    download = INPUT_H5.with_suffix(".download")
    request = Request(DATA_URL, headers={"Range": "bytes=0-", "User-Agent": "Mozilla/5.0"})
    with urlopen(request) as source, download.open("wb") as destination:
        copyfileobj(source, destination)
    download.replace(INPUT_H5)

raw = sc.read_10x_h5(INPUT_H5)
print(f"Downloaded matrix: {raw.n_obs:,} barcodes × {raw.n_vars:,} features")


In [ ]:
human_genes = raw.var["genome"].eq("GRCh38").to_numpy()
mouse_genes = raw.var["genome"].eq("GRCm39").to_numpy()
human_umis = np.asarray(raw[:, human_genes].X.sum(axis=1)).ravel()
mouse_umis = np.asarray(raw[:, mouse_genes].X.sum(axis=1)).ravel()
total_umis = human_umis + mouse_umis
human_cells = human_umis / total_umis >= 0.9
mouse_cells = mouse_umis / total_umis >= 0.9


def collapse_species(cell_mask, gene_mask, prefix):
    subset = raw[cell_mask, gene_mask]
    symbols = pd.Index(subset.var_names.str.removeprefix(prefix), name="gene_symbol")
    gene_ids = subset.var["gene_ids"].astype(str).str.removeprefix(prefix)
    unique_symbols = symbols.drop_duplicates()
    output_index = pd.Series(np.arange(len(unique_symbols)), index=unique_symbols)
    columns = output_index.loc[symbols].to_numpy()
    projection = sparse.csr_matrix(
        (np.ones(len(symbols), dtype=np.int32), (np.arange(len(symbols)), columns)),
        shape=(len(symbols), len(unique_symbols)),
    )
    counts = (subset.X.tocsr().astype(np.int32) @ projection).tocsr()
    ids = pd.Series(gene_ids.to_numpy(), index=symbols).groupby(level=0, sort=False).agg(";".join)
    var = pd.DataFrame({"gene_ids": ids.loc[unique_symbols].to_numpy()}, index=unique_symbols)
    return counts, var, subset.obs_names.copy()


collapsed = {
    "human": collapse_species(human_cells, human_genes, "GRCh38_"),
    "mouse": collapse_species(mouse_cells, mouse_genes, "GRCm39_"),
}
shared_genes = sorted(
    {gene.casefold() for gene in collapsed["human"][1].index}
    & {gene.casefold() for gene in collapsed["mouse"][1].index}
)


def normalize_species(species, cell_line):
    counts, var, obs_names = collapsed[species]
    lookup = {gene.casefold(): index for index, gene in enumerate(var.index)}
    shared_indices = np.asarray([lookup[gene] for gene in shared_genes])
    library_size = np.asarray(counts[:, shared_indices].sum(axis=1)).ravel()
    expression = counts.astype(np.float32).multiply((10_000 / library_size)[:, None]).tocsr()
    np.log1p(expression.data, out=expression.data)
    obs = pd.DataFrame({"cell_line": cell_line, "species": species}, index=obs_names)
    result = ad.AnnData(expression, obs=obs, var=var)
    result.layers["counts"] = counts
    if species == "mouse":
        dispersion_input = ad.AnnData(counts[:, shared_indices].copy(), var=var.iloc[shared_indices].copy())
        sc.pp.log1p(dispersion_input)
        sc.pp.highly_variable_genes(dispersion_input)
        dispersions = pd.Series(
            dispersion_input.var["dispersions_norm"].to_numpy(),
            index=dispersion_input.var_names.str.casefold(),
        )
        result.var["dispersions_norm"] = [dispersions.get(gene.casefold(), np.nan) for gene in var.index]
    return result


adata_human = normalize_species("human", "HEK293T")
adata_mouse = normalize_species("mouse", "NIH3T3")
print(f"Human: {adata_human.n_obs:,} cells; mouse: {adata_mouse.n_obs:,} cells")
print(f"Mixed or ambiguous barcodes removed: {(~human_cells & ~mouse_cells).sum():,}")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for label, mask, color in (
    ("Human", human_cells, "#4b9cbe"),
    ("Mouse", mouse_cells, "#7a9a01"),
    ("Mixed", ~human_cells & ~mouse_cells, "#999999"),
):
    ax.scatter(human_umis[mask] + 1, mouse_umis[mask] + 1, s=5, alpha=0.35, label=label, color=color)
ax.set(xscale="log", yscale="log", xlabel="Human UMIs + 1", ylabel="Mouse UMIs + 1", title="Species assignment")
ax.legend()
plt.tight_layout()


In [ ]:
STEPS_PER_BATCH = 250
EPOCHS = 29
POSTERIOR_SAMPLES = 2_000
TRAINING_SEED = 20260717
POSTERIOR_SEED = 20260718
VALIDATION_CELLS = 687
VALIDATION_SPLIT_SEED = 20260718
VALIDATION_SEED = 20260719

rng = np.random.RandomState(VALIDATION_SPLIT_SEED)
held_out_indices = rng.choice(adata_mouse.n_obs, VALIDATION_CELLS, replace=False)
rng.shuffle(held_out_indices)
training_indices = np.setdiff1d(np.arange(adata_mouse.n_obs), held_out_indices)

network = xc.prepare_network(adata_mouse, adata_human, dispersion_cutoff=-10)
ligand_abundance = xc.compute_ligand_abundance(
    adata_mouse,
    adata_human,
    network["ligands"],
    network["human_ligands"],
    network["ligand_receptor_matrix"],
)
model = xc.XenocommModel(
    adata_mouse[training_indices],
    **network,
    mean_ligand=ligand_abundance,
    receptor_target_mode="learned",
    training_seed=TRAINING_SEED,
    posterior_seed=POSTERIOR_SEED,
    batch_size=687,
    steps_per_batch=STEPS_PER_BATCH,
    epochs=EPOCHS,
)
training = model.train(
    validation_mouse=adata_mouse[held_out_indices],
    absolute_tolerance=0.001 * 687,
    patience=3,
    min_evaluations=5,
    validation_seed=VALIDATION_SEED,
)
training


In [ ]:
samples = model.sample(POSTERIOR_SAMPLES)
parameters = model.get_parameters()
ligand_results = xc.ligand_result_table(
    model.ligands,
    samples,
    model.mean_ligand_np,
    model.ligand_receptor_matrix_np,
    xc.get_receptor_sensitivity(parameters),
)

payload = {
    "ligands": np.asarray(model.ligands),
    "human_ligands": np.asarray(model.human_ligands),
    "receptors": np.asarray(model.receptors),
    "targets": np.asarray(model.targets),
    "mean_ligand_np": model.mean_ligand_np,
    "ligand_receptor_matrix_np": model.ligand_receptor_matrix_np,
    **{f"s_{key}": value for key, value in samples.items()},
    **{f"v_{key}": value for key, value in parameters.items()},
}
np.savez(OUTPUT_DIR / "model.staged.npz", **payload)
ligand_results.to_parquet(OUTPUT_DIR / "ligand_results.parquet", index=False)

detected = ligand_results.loc[ligand_results["called"]].head(15).copy()
print(f"Detected ligands: {len(ligand_results.loc[ligand_results['called']])}")
display(detected[["ligand", "delta_h_mean", "human_fraction_mean"]])


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=detected, x="delta_h_mean", y="ligand", ax=ax, color="#4b9cbe")
ax.set(xlabel="Human counterfactual activation", ylabel="Ligand", title="Detected ligands")
plt.tight_layout()


In [ ]:
binding = xc.species_bias_df(model, samples, parameters)
binding = binding[binding["ligand"].isin(detected["ligand"])].melt(
    id_vars="ligand",
    value_vars=["mouse_binding", "human_binding"],
    var_name="species",
    value_name="binding",
)
binding["species"] = binding["species"].str.replace("_binding", "", regex=False).str.title()
binding["ligand"] = pd.Categorical(binding["ligand"], detected["ligand"], ordered=True)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=binding, x="binding", y="ligand", hue="species", ax=ax)
ax.set(xlabel="Binding score", ylabel="Ligand", title="Species-decomposed ligand binding")
plt.tight_layout()


In [ ]:
receptors = xc.receptor_marginal_df(model, samples, parameters).nlargest(12, "delta")
receptors_long = receptors.melt(
    id_vars="receptor", value_vars=["mouse", "delta"], var_name="component", value_name="activation"
)
receptors_long["component"] = receptors_long["component"].map(
    {"mouse": "Mouse baseline", "delta": "Human delta"}
)
receptors_long["receptor"] = pd.Categorical(
    receptors_long["receptor"], receptors["receptor"], ordered=True
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=receptors_long, x="activation", y="receptor", hue="component", ax=ax)
ax.set(xlabel="Activation", ylabel="Receptor", title="Top receptor marginal activation")
plt.tight_layout()


In [ ]:
top_ligand = detected.iloc[0]["ligand"]
counterfactual = xc.receptor_counterfactual_df(
    model, samples, parameters, remove_ligands=[top_ligand], label=top_ligand
)
pivot = counterfactual.pivot(index="receptor", columns="component", values="activation").fillna(0)
pivot = pivot.loc[pivot.sum(axis=1).nlargest(12).index]

fig, ax = plt.subplots(figsize=(8, 4))
pivot.plot.barh(stacked=True, ax=ax)
ax.invert_yaxis()
ax.set(xlabel="Receptor activation", ylabel="Receptor", title=f"Counterfactual receptor activation: {top_ligand}")
plt.tight_layout()


In [ ]:
targets = xc.targets_marginal_df(model, samples, parameters, top_n=15)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=targets, x="value", y="target", ax=ax, color="#7a9a01")
ax.set(xlabel="Weighted activation", ylabel="Target", title="Top downstream targets")
plt.tight_layout()
targets


In [ ]:
dot_data = xc.species_dotplot_data(
    model, adata_mouse, adata_human, ligands=detected["ligand"].head(8)
)
dot = pd.DataFrame(dot_data)
fig, ax = plt.subplots(figsize=(7, 3.5))
points = ax.scatter(dot["genes"], dot["species"], s=np.asarray(dot["size"]) * 500, c=dot["color"], cmap="viridis")
ax.set(xlabel="Ligand", ylabel="Species", title="Cross-species ligand expression")
plt.xticks(rotation=45, ha="right")
plt.colorbar(points, ax=ax, label="Mean expression")
plt.tight_layout()
